# Module 6 - Session 5: Practical Exercises

**Total Time Estimate:** 90-120 minutes

**Objective:** To build a simple, functional retrieval-based chatbot from scratch, solidifying the concepts of TF-IDF and cosine similarity.

## Setup

You will need Python with `scikit-learn` and `nltk`.

```bash
pip install scikit-learn nltk
```


## Exercise 1: Conceptual Questions (20 minutes)

### Foundation

Retrieval-based chatbots do not generate new text from scratch. They select the best answer from a fixed knowledge base, which makes them easier to control but also limits what they can say.

### Build

1. **Choosing the Right Bot**

For a bank website that answers short factual questions, I would choose a **retrieval-based chatbot**. It is safer and more controllable than a generative system because it only answers from approved content, and it is more flexible than a purely rule-based bot because it can still handle slightly rephrased user questions. The main advantage is reliability and lower risk of hallucination, while the main limitation is that it cannot answer questions outside the knowledge base or handle complex multi-step reasoning.

2. **The Importance of the Corpus**

The user's query must be transformed with the **same fitted TF-IDF vectorizer** so that the query and the knowledge-base questions live in the same feature space and use the same IDF weights. If you calculated a new TF-IDF vector for the query in isolation, the weighting would no longer be comparable because the query would be treated as its own tiny corpus. That would distort term importance and make cosine similarity with the stored FAQ matrix meaningless or at least inconsistent.

3. **Failure Modes**

From a TF-IDF perspective, `Is the food in the cafeteria good?` may match `Where is the cafeteria located?` because both sentences share the rare content word `cafeteria`, which can dominate the similarity score when the knowledge base is small. It is a terrible user experience because lexical overlap is not the same as semantic intent: the user asked for an opinion about food quality, but the bot answered a location question. This is exactly why retrieval systems need confidence thresholds and, in stronger systems, better semantic representations.

### Result

Retrieval-based bots work best when the domain is narrow, the answers are factual, and the knowledge base is curated carefully. Their main weakness is that surface-level word overlap can still produce bad matches.


## Exercise 2: Building a Simple FAQ Bot (70 minutes)

In this exercise, you will build the core logic for a retrieval-based chatbot using TF-IDF and cosine similarity.


In [5]:
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

faq_database = {
    'What are the admission requirements?': 'You need a high school diploma and must pass the entrance exam.',
    'When is the application deadline?': 'The application deadline is July 31st.',
    'What time does the library close?': 'The library closes at 9 PM on weekdays and 5 PM on weekends.',
    'How can I get a parking permit?': 'You can apply for a parking permit through the student portal.',
    'Where is the registrar office located?': 'The registrar office is in the Administration Building, Room 102.',
    'How do I reset my student portal password?': 'Use the Forgot Password link on the student portal login page.',
    'Do you offer on-campus housing?': 'Yes, on-campus housing is available for both first-year and returning students.'
}

questions = list(faq_database.keys())
answers = list(faq_database.values())

vectorizer = TfidfVectorizer(stop_words='english')
faq_matrix = vectorizer.fit_transform(questions)

print(f'Number of FAQ entries: {len(questions)}')
print(f'TF-IDF matrix shape: {faq_matrix.shape}')


Number of FAQ entries: 7
TF-IDF matrix shape: (7, 20)


In [6]:
def get_response(query, threshold=0.0):
    query_vector = vectorizer.transform([query])
    similarities = cosine_similarity(query_vector, faq_matrix)[0]

    best_match_index = int(np.argmax(similarities))
    best_score = float(similarities[best_match_index])
    best_question = questions[best_match_index]

    if best_score < threshold:
        return {
            'query': query,
            'matched_question': None,
            'score': best_score,
            'answer': "I'm sorry, I don't have an answer for that. Please try rephrasing your question."
        }

    return {
        'query': query,
        'matched_question': best_question,
        'score': best_score,
        'answer': faq_database[best_question]
    }


In [7]:
test_queries = [
    'How do I apply for admission?',
    'When is the deadline to apply?',
    'What time does the library close on weekdays?',
    'How can I get a parking pass?',
    'Where is the registrar office?',
    'I need to reset my student portal password.',
    'Is there housing on campus?'
]

for query in test_queries:
    result = get_response(query)
    print(f'User: {query}')
    print(f"Matched FAQ: {result['matched_question']}")
    print(f"Similarity score: {result['score']:.4f}")
    print(f"Bot: {result['answer']}")
    print('-' * 80)


User: How do I apply for admission?
Matched FAQ: What are the admission requirements?
Similarity score: 0.7071
Bot: You need a high school diploma and must pass the entrance exam.
--------------------------------------------------------------------------------
User: When is the deadline to apply?
Matched FAQ: When is the application deadline?
Similarity score: 0.7071
Bot: The application deadline is July 31st.
--------------------------------------------------------------------------------
User: What time does the library close on weekdays?
Matched FAQ: What time does the library close?
Similarity score: 1.0000
Bot: The library closes at 9 PM on weekdays and 5 PM on weekends.
--------------------------------------------------------------------------------
User: How can I get a parking pass?
Matched FAQ: How can I get a parking permit?
Similarity score: 0.7071
Bot: You can apply for a parking permit through the student portal.
------------------------------------------------------------

### Analysis (Markdown Answer)

### Foundation

The chatbot works by comparing the user's question vector against all FAQ question vectors and choosing the most similar one.

### Build

This simple system performs well when the user's wording overlaps clearly with the FAQ wording, as in pairs like `parking pass` versus `parking permit` or `registrar office` versus `registrar office located`. Because the vectorizer is fitted only on the FAQ questions, the representation stays small and efficient, which is ideal for a lightweight retrieval bot.

At the same time, the model is still lexical rather than deeply semantic. If a user asks with very different vocabulary such as `Do students have dorms?`, the match can weaken even when the intent is close to `on-campus housing`. That is the central trade-off in a basic TF-IDF chatbot: it is transparent and easy to build, but it depends heavily on word overlap.

### Result

A retrieval-based FAQ bot can be built with very little code, and it already works surprisingly well for narrow domains with predictable phrasing.


## Exercise 3: Challenge Problem - Adding a Confidence Threshold (30 minutes)

### Foundation

A retrieval bot should not answer every question. If the top similarity score is too low, returning a fallback message is safer than giving a confidently wrong answer.

### Build

The `get_response(query, threshold=0.0)` function already supports a threshold parameter. Here we test it with `threshold=0.7` on both relevant and irrelevant questions.

### Result

Run the next cell to see how the bot behaves once low-confidence matches are blocked.


In [8]:
threshold_test_queries = [
    'How can I get a parking pass?',
    'Where is the registrar office?',
    'What is the meaning of life?',
    'Is cafeteria food good?'
]

for query in threshold_test_queries:
    result = get_response(query, threshold=0.7)
    print(f'User: {query}')
    print(f"Matched FAQ: {result['matched_question']}")
    print(f"Similarity score: {result['score']:.4f}")
    print(f"Bot: {result['answer']}")
    print('-' * 80)


User: How can I get a parking pass?
Matched FAQ: How can I get a parking permit?
Similarity score: 0.7071
Bot: You can apply for a parking permit through the student portal.
--------------------------------------------------------------------------------
User: Where is the registrar office?
Matched FAQ: Where is the registrar office located?
Similarity score: 0.8165
Bot: The registrar office is in the Administration Building, Room 102.
--------------------------------------------------------------------------------
User: What is the meaning of life?
Matched FAQ: None
Similarity score: 0.0000
Bot: I'm sorry, I don't have an answer for that. Please try rephrasing your question.
--------------------------------------------------------------------------------
User: Is cafeteria food good?
Matched FAQ: None
Similarity score: 0.0000
Bot: I'm sorry, I don't have an answer for that. Please try rephrasing your question.
---------------------------------------------------------------------------

### Analysis (Markdown Answer)

### Foundation

The threshold controls the balance between answering more often and answering more safely.

### Build

With a threshold such as **`0.7`**, strongly overlapping questions like `How can I get a parking pass?` still receive the correct answer, while clearly unrelated questions such as `What is the meaning of life?` fall back safely. This improves the user experience because the bot is no longer forced to give a misleading answer just because one FAQ entry is mathematically the least bad match.

The trade-off is that some relevant but more loosely phrased questions may also be rejected if their wording differs too much from the FAQ database. In practice, you tune the threshold based on real user queries and decide whether you care more about recall or precision.

### Result

Confidence thresholds are a simple but important upgrade. They turn a naive retrieval system into a safer and more realistic chatbot for real users.
